<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Accessing and Exploring Clinical Data
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

Today is mostly yours: a short run-through of MIMIC-IV and how to query it, then a long Applied Challenge on real ICU data. Three kinds of activity are marked with **»**:

* **» Investigate** — run a working cell, then answer a question about it, from the output or by changing one thing.
* **» Modify** — change a working cell so it answers the next question.
* **» Make** — write the code yourself. The pattern is always somewhere above. **Hint** blocks fold open if you need them: try first.

This follows the PRIMM pattern used in programming teaching: Predict, Run, Investigate, Modify, Make.

Part way through, an **Independent practice** slide hands over to you for the Applied Challenge; a **Regroup** slide marks where we come back together.

The notebook is **one environment**: the imports, the `query` function and every table you load exist in every cell after them, so the cells are meant to be run in order. After a restart, or if a cell complains that `query` or a `df_...` is not defined, use **Runtime → Run before** to catch up.

**MIMIC-IV outputs are not in the published slides or notes.** The data use agreement forbids sharing the data, so you see them only when you run the cells yourself. For the same reason, never paste MIMIC rows into an AI assistant or share them; asking about your code is fine.

> Worked solutions, with explanations, are in the solutions notebook, released after the session.

## Welcome

**Goal:** by the end of today you will query a real ICU database from Python, build a patient cohort from it, and plot what you find.

Sessions 1 and 2 used small teaching files. Today is the real thing: **MIMIC-IV**, hundreds of thousands of patients, through the PhysioNet access you set up. Your presentations come first, then a short run-through of how to query MIMIC, then about 70 minutes of Applied Challenge in pairs. If you are stuck, say so.

### Before we start
1. Open this notebook in Colab: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/03_Accessing_and_Exploring_Clinical_Data.ipynb).
2. **File → Save a copy in Drive**, and work in your copy.
3. Run the **access check** in section 1 now, so problems surface early. If one of you cannot get in, work on your partner's screen, and fix your access after the session.

## Intended Learning Outcomes
By the end of this session you will be able to:

1. **Describe** what MIMIC-IV contains, and **identify** the features of its data that need care: shifted dates, capped ages, two diagnosis coding systems.
2. **Query** a clinical database from Python with `SELECT * FROM`, and **explain** what `LIMIT`, `WHERE` and a view do to the rows you get.
3. **Apply** loading, filtering, merging and plotting to answer questions about a real ICU cohort.
4. **Justify** a cohort definition, and **identify** which data are available at a given prediction time.

## Session outline
1. Check your access
2. Your MIMIC presentations, then MIMIC-IV in brief
3. Querying MIMIC from Python
4. Independent practice: Applied Challenge
5. Closing

# 1. Check your access
Run the cell below. The first time, Colab asks you to sign in: use the **Google account linked to PhysioNet**. The cell then runs a one-row query and prints either **Access OK** or what to fix.

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import matplotlib.pyplot as plt
import pandas_gbq

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

try:
    pandas_gbq.read_gbq("SELECT 1 FROM `physionet-data.mimiciv_3_1_hosp.patients` LIMIT 1",
                        project_id=PROJECT)
    print('Access OK')
except Exception as error:
    message = str(error)
    if 'jobs.create' in message:
        print('Not in the course project yet: send me this Google address (see the next slide).')
    elif 'physionet-data' in message:
        print('No MIMIC-IV access for this Google account: link it in PhysioNet (see the next slide).')
    else:
        print('Something else went wrong. Show me this message:')
        print(message)

## If the access check fails
**Full instructions are on Canvas.**

* **No MIMIC-IV access:** request access to MIMIC-IV 3.1 through BigQuery, using the link in the Files section of the PhysioNet page, [physionet.org/content/mimiciv/3.1/#files](https://physionet.org/content/mimiciv/3.1/#files). If there is no 'Request access using Google BigQuery' option, check you have added and enabled your Gmail address in PhysioNet. The reply looks like: *Access to the GCP BigQuery has been granted to your.email@gmail.com for project: MIMIC-IV v3.1*
* **Not in the course project:** send me that Gmail address (see Canvas), so I can add you to `bowyerlab-mimic-teaching`.
* **Signed in with the wrong Google account:** **Runtime → Disconnect and delete runtime**, then run the cell again and choose the PhysioNet-linked account.
* You do not need your own Google Cloud project during the course, and you never need to open BigQuery itself: we query it from Colab. After the course, set `PROJECT` to a Google Cloud project of your own.

# 2. MIMIC-IV
## Your MIMIC tables
**Presentations: 4 minutes per group, from your Miro frame (link on Canvas).**

* **Your tables:** what one row is, and how a row links to a patient, an admission and a time.
* **Your recipe:** the feature, step by step, and which rows you could use at `prediction_time`.

Listen for the links between tables: the rest of today joins them.

## Medical Information Mart for Intensive Care
[www.nature.com/articles/s41597-022-01899-x](https://www.nature.com/articles/s41597-022-01899-x)

* Large, freely available database of de-identified health data from patients admitted to the Beth Israel Deaconess Medical Center ICU in Boston, USA
* Focused on ICU, but version IV includes hospital-wide data
* Used for learning, clinical informatics, clinical decision making, and ML
* Data from:
    * **364,627 unique patients**
    * **546,028 admissions**
    * **94,458 ICU stays**

## Data contents

* **Patient demographics:** age, gender, admission and discharge information
* **Clinical observations:** vital signs, lab results
* **Diagnoses:** ICD codes
* **Interventions:** procedures, medications
* **Outcome data:** length of stay, mortality
* **Free-text notes:** nursing notes, discharge summaries (a separate module, MIMIC-IV-Note, not used in this course)

## Challenges of using MIMIC

* Calendar dates are shifted into the future (e.g. "2150-05-12")
    * Consistent within a patient
    * Unusable between patients
* Ages over 89 are shown as 91
* Text data are redacted
* All the challenges of real data: missingness, formatting issues, alignment
* 100+ GB of data

You will find the first two in the data yourselves, in section 3.

## BigQuery: a cloud database

* Google Cloud's data warehouse: run SQL on huge datasets
* No servers to manage: fully managed and scalable
* Fast: queries on millions of records in seconds
* Pay-as-you-go: pay only for storage and queries
* Works with Python and notebooks
* Well suited to healthcare data: secure, reproducible, and handles large EHR datasets

# 3. Querying MIMIC from Python
## Database queries (SQL)

* Database structures and query languages are a topic of their own
* In this course, all data manipulation is in Python; SQL only gets the data out
* Structured Query Language (SQL) is the standard language for databases
* The one statement you need: `SELECT * FROM table` returns every column (`*`) of every row of `table`

## From SQL to Python

* We run SQL from Python with the `pandas_gbq` library [[github link]](https://github.com/googleapis/python-bigquery-pandas), so the result arrives as a pandas DataFrame
* `query` below wraps it in a function, so each query is one line
* The query returns the first 100 rows of the `patients` table: `LIMIT 100` stops after 100 rows
* What comes back is an ordinary pandas DataFrame. `subject_id` identifies the patient in every MIMIC table about patients, so it is how the tables join

In [ ]:
def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

df_first = query("SELECT * FROM `physionet-data.mimiciv_3_1_hosp.patients` LIMIT 100")
df_first.head()

## The whole table
Drop the `LIMIT` and you get every row: all 364,627 patients, in about 30 seconds.

What comes back is an ordinary DataFrame, so everything from Session 2 works on it. One use: checking the data are what you think they are, before you analyse them.

In [ ]:
df_all_patients = query("SELECT * FROM `physionet-data.mimiciv_3_1_hosp.patients`")
len(df_all_patients)

The cell below uses `.max()` and `.min()` to check two warnings from *Challenges of using MIMIC*: the oldest `anchor_age`, and the years `anchor_year` covers. The [patients documentation](https://mimic.mit.edu/docs/iv/modules/hosp/patients/) explains both columns.

In [ ]:
print(df_all_patients['anchor_age'].max())
print(df_all_patients['anchor_year'].min(), df_all_patients['anchor_year'].max())

## LIMIT returns the first rows, not a sample
How many atrial fibrillation (AF) diagnoses are there? In ICD-9, the code for AF is `427.31`, stored in MIMIC without the dot.

In [ ]:
df_some_diagnoses = query("SELECT * FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` LIMIT 10000")
print((df_some_diagnoses['icd_code'] == '42731').sum())

**» Investigate** — **Change `10000` to `100000` and run the cell again.** What happened to the count? The full table has 6.4 million rows: is either count the number of AF diagnoses in MIMIC, or a known fraction of it? Why not?

## Views: saved queries

* A **view** is a `SELECT` saved in the database under a name. You query it like a table, and the saved `SELECT` runs underneath.
* For this course I have saved views over a fixed sample: **1 in 10 of the MIMIC-IV patients who had an ICU stay**, with their ICU admissions. That is 6,549 patients and 8,647 admissions: small enough to load whole, large enough to be worth analysing.
* They are in `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views`: `cohort` (the patients and admissions in the sample), `patients`, `admissions`, `diagnoses_icd`, `d_items` (the dictionary of ICU measurements) and `chartevents_hr` (heart rate readings).
* They hold no data: querying a view still needs your own PhysioNet access. Their definitions are in the course repository, [sql/views](https://github.com/stuartbowyer/sdsai-lecture-notes/tree/main/sql/views/sdsai_mimiciv_icu_small_views).

**» Investigate** — Read a view. `chartevents_hr` is this saved query:

```sql
SELECT hadm_id, charttime, itemid, valuenum
FROM `physionet-data.mimiciv_3_1_icu.chartevents`
WHERE itemid = 220045
  AND hadm_id IN (SELECT hadm_id FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.cohort`)
```

1. **Which line keeps only heart rate?**
2. **Which line keeps only our cohort's admissions?**
3. **Why four named columns instead of `*`?** `chartevents` has 433 million rows.

## The cohort's diagnoses
No `LIMIT`: every diagnosis for every admission in the cohort, chosen by patient rather than by where the rows happen to sit in the table. You will use `df_diagnoses` in the Applied Challenge.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
print(len(df_diagnoses))
print((df_diagnoses['icd_code'] == '42731').sum())

# 4. Independent practice: Applied Challenge
From here to the **Closing** is yours, about 70 minutes, in your pair: three parts on the cohort, from loading tables to a sepsis cohort.

* Each task is marked **» Modify** or **» Make**.
* **Hint** blocks fold open: try first.
* Every table comes from the course views with `SELECT * FROM`; no `LIMIT` needed.
* Run the cells in order: each part builds on the one before.

Work together in your pair, and call me over when you get stuck.

**Understanding beats finishing.**

## Part 1: Who is in the cohort?
The cohort's patients, one row each.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
print(len(df_patients))
df_patients.head()

**» Modify** — Change the cell below so it loads the **`admissions`** view into **`df_admissions`**. The full MIMIC-IV table is `physionet-data.mimiciv_3_1_hosp.admissions`, and the [admissions documentation](https://mimic.mit.edu/docs/iv/modules/hosp/admissions/) says what each column is.

In [ ]:
# Change the view and the variable name: load admissions into df_admissions
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
print(len(df_patients))

**» Modify** — This counts patients by `anchor_year_group`, the three-year window each patient's real dates fall in. **Change it to count patients by `gender`**, then add `normalize=True` inside the brackets to get proportions instead of counts.

In [ ]:
df_patients['anchor_year_group'].value_counts()

**» Make** — **Plot the gender split** as a bar chart, applying *Before you hand in a figure* from Session 2.

<details><summary><b>Hint</b></summary>

`value_counts()` returns a Series, so `.plot(kind='bar')` goes straight on the end. Then `plt.title(...)` and `plt.ylabel(...)`.

</details>

In [ ]:
# Your code here

## Part 2: Atrial fibrillation
The diagnosis table from section 3. `subject_id` links each diagnosis to a patient, and `hadm_id` to an admission, and so to a time. `icd_code` is the diagnosis, in the coding system named by `icd_version`. See the [diagnoses_icd documentation](https://mimic.mit.edu/docs/iv/modules/hosp/diagnoses_icd/).

In [ ]:
df_diagnoses.head()

Diagnoses are coded in two systems, ICD-9 and the newer ICD-10:

In [ ]:
df_diagnoses['icd_version'].value_counts()

**» Modify** — The AF count in section 3 used `42731`, an ICD-9 code, so it missed every AF diagnosis coded in ICD-10, where AF codes start with `I48`. Change the cell below so `df_af` keeps the **ICD-10 AF codes as well**. How much does the count change?

<details><summary><b>Hint</b></summary>

`df_diagnoses['icd_code'].str.startswith('I48')` is `True` for the ICD-10 codes. Two conditions combine with `|` (or), each in its own brackets: `df[(condition_1) | (condition_2)]`.

</details>

In [ ]:
df_af = df_diagnoses[df_diagnoses['icd_code'] == '42731']
len(df_af)

**» Make** — **How many patients have an AF diagnosis?** Compare it with the number of AF diagnoses: why are they different?

<details><summary><b>Hint</b></summary>

`.nunique()` counts the distinct values in a column. Which column identifies a patient?

</details>

In [ ]:
# Your code here

**» Make** — **The gender split of patients with AF**, as proportions. How does it compare with the whole cohort from Part 1?

<details><summary><b>Hint</b></summary>

Keep the rows of `df_patients` whose `subject_id` is in `df_af['subject_id']`, with `.isin(...)`. Or `merge` `df_patients` with the distinct AF patients: `df_af[['subject_id']].drop_duplicates()`. Then the same `value_counts` as Part 1.

</details>

In [ ]:
# Your code here

## Part 3: A sepsis cohort
A cohort is a definition you choose, and must be able to defend. Here: ICU admissions with sepsis.

| System | Code in MIMIC | Meaning |
| --- | --- | --- |
| ICD-9 | `99591` | Sepsis |
| ICD-9 | `99592` | Severe sepsis |
| ICD-9 | `78552` | Septic shock |
| ICD-10 | starts with `A41` | Sepsis, by organism |
| ICD-10 | `R6520` | Severe sepsis without septic shock |
| ICD-10 | `R6521` | Severe sepsis with septic shock |

**» Make** — **Decide which codes count as sepsis**, and write your choice in a comment: it is your cohort definition. Then make **`df_sepsis`**, with **one row per admission** and the columns `subject_id` and `hadm_id`.

<details><summary><b>Hint</b></summary>

1. `df_diagnoses['icd_code'].isin([...])` is `True` for any code in the list; combine it with `.str.startswith('A41')` using `|`, as in Part 2.
2. An admission can have several sepsis codes. `df[['subject_id', 'hadm_id']].drop_duplicates()` keeps one row per admission.
3. Check: depending on the codes you chose, somewhere between about 880 and 1,430 admissions.

</details>

In [ ]:
# My sepsis definition:

# Your code here

**» Make** — **Their admissions.** Merge `df_sepsis` with `df_admissions` into **`df_sepsis_admissions`**, so each sepsis admission has its `admittime` and `dischtime`. Check the number of rows has not changed.

<details><summary><b>Hint</b></summary>

Both tables have `subject_id` **and** `hadm_id`. Merge on both, `on=['subject_id', 'hadm_id']`: merging on one leaves two copies of the other, renamed `subject_id_x` and `subject_id_y`.

</details>

In [ ]:
# Your code here

**» Make** — **The gender split of the sepsis cohort.** Is it different from the whole cohort?

<details><summary><b>Hint</b></summary>

Merge `df_sepsis_admissions` with `df_patients` on `subject_id`, then `value_counts(normalize=True)` as before.

</details>

In [ ]:
# Your code here

**» Make** — **Length of stay.** Add a column with each sepsis admission's length of stay **in days**, and plot its distribution. What is the median?

<details><summary><b>Hint</b></summary>

`admittime` and `dischtime` arrive from BigQuery as datetimes, not text, so `dischtime - admittime` works directly and gives a timedelta. `.dt.total_seconds()` turns it into seconds; divide by `60 * 60 * 24` for days. The tail is long: try `bins=50`, then `plt.xlim(0, 60)`.

</details>

In [ ]:
# Your code here

## Heart rate
The `chartevents_hr` view keeps `itemid` 220045. Raw MIMIC identifies every ICU measurement only by an `itemid`; `d_items` is the dictionary that says what each one is, and in what units.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.d_items
df_items = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.d_items`")
df_items[df_items['itemid'] == 220045]

Every heart rate reading for the cohort's admissions: about 900,000 rows, so it takes a minute. **Run it once.**

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.chartevents, keeping itemid 220045
df_hr = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.chartevents_hr`")
len(df_hr)

**» Make** — **One admission's heart rate.** Find the sepsis admission with the **most** heart rate readings, and plot its heart rate against time.

<details><summary><b>Hint</b></summary>

1. Keep the rows of `df_hr` whose `hadm_id` is in `df_sepsis_admissions['hadm_id']`, with `.isin(...)`.
2. `['hadm_id'].value_counts()` on those rows puts the admission with the most readings first; `.index[0]` is its `hadm_id`.
3. Keep that admission's rows, `sort_values('charttime')`, then `.plot(x='charttime', y='valuenum')`.

</details>

In [ ]:
# Your code here

**» Make** — **Heart rate across the sepsis cohort.** Plot the distribution of every heart rate reading for the sepsis admissions.

If your histogram is one tall bar, run `.describe()` on the values: what is the largest, and is it a heart rate? Plot again keeping only plausible values, and write in a comment which limits you chose and why.

<details><summary><b>Hint</b></summary>

The rows from step 1 of the previous task are the readings you need; plot their `valuenum` column with `.hist(bins=50)`. Two conditions keep a range: `(values >= low) & (values <= high)`.

</details>

In [ ]:
# Your code here

**» Make (stretch)** — **Is sepsis deadly here?** In `admissions`, `hospital_expire_flag` is 1 if the patient died in hospital and 0 if not. **Compare in-hospital mortality for your sepsis admissions with the rest of the cohort's admissions.** Then, if you have time: does the rate change much with a different sepsis definition?

<details><summary><b>Hint</b></summary>

1. `df_sepsis_admissions` already has `hospital_expire_flag`, from its merge with `df_admissions`. The mean of a 0/1 column is the proportion of 1s.
2. The rest of the cohort: the rows of `df_admissions` whose `hadm_id` is **not** in `df_sepsis['hadm_id']`. `~` in front of a condition flips it: `df[~df['hadm_id'].isin(...)]`.

</details>

In [ ]:
# Your code here

**» Make (stretch)** — **Sepsis from blood cultures.** ICD codes are assigned for billing, after discharge. A definition from the stay itself: admissions in which a **blood culture** was taken, a test ordered when infection is suspected.

The cell below queries the full `microbiologyevents` table, with two `WHERE` conditions like those in `chartevents_hr`: blood cultures only, for the cohort's admissions only.

1. **How many cohort admissions had a blood culture?**
2. **How many of your ICD sepsis admissions had one?** And how many blood culture admissions have no sepsis code?
3. **Which definition would you trust for a study of sepsis outcomes, and why?**

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.microbiologyevents (no course view)
df_cultures = query("""
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents`
WHERE spec_type_desc = 'BLOOD CULTURE'
  AND hadm_id IN (SELECT hadm_id FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.cohort`)
""")

# Your code here

## Regroup

Stop where you are: we come back together for the Closing.

If you didn't finish, finish after the session: Part 3 is the most valuable, and the stretch tasks are optional.

Finished early? Help a neighbour.

# 5. Closing
## What your numbers depend on
In your pair for two minutes, then together:

1. **Definition.** Even with both coding systems, "sepsis" can mean different things. Any sepsis code: about 1,430 admissions, 29% of whom died in hospital. Severe sepsis or septic shock only: about 1,110, 32%. Septic shock only: about 880, 34%. Which would you use for a study of sepsis outcomes, and why? Did your pair choose the same as the pair next to you?
2. **Time.** Your heart rate histogram used every reading in each admission. Take the `prediction_time` from your Miro recipe: if a model had to flag sepsis then, which readings could it use? Diagnosis codes are assigned for billing, at discharge: could it use those?
3. **Population.** Every number today comes from ICU patients at one hospital in Boston. Your sepsis mortality of about 29%: who could you quote it for?

Session 4 starts from the second: using only what is known at prediction time.

## Wrap up
* First coursework is set today: check the due date on Canvas.
* The course views are a starting point. The full MIMIC-IV tables are in `physionet-data`, queried in exactly the same way: the comment above each view query names the full table.
* MIMIC data stay in your notebooks: do not share outputs, and do not paste rows into AI assistants.
* Next week we start machine learning, with preparing data for it.

## Before Session 4
1. **Finish the Applied Challenge**, including the stretch tasks, then check your answers against the solutions notebook once it is released.
2. **Catch up on the technical tutorials from Sessions 1 to 3**, including any tasks you did not reach: Session 4 builds on all of them.
3. **Work through the Session 4 preparation notebook**, on [Canvas](https://canvas.imperial.ac.uk/courses/8306): data preparation methods, practised on the course MIMIC views. Session 4 assumes you have done it and does not re-teach the methods.
4. Check the reading list.
5. Coursework